# Breast Cancer classification with SeeDL

Fit a scikit-learn pipeline explicitly and inspect held-out classification metrics with `sd.evaluate`. This is an educational benchmark, **not a clinical validation**. The authentic Wisconsin Diagnostic Breast Cancer dataset is bundled with scikit-learn; no dataset download is needed.

Install this checkout with `python -m pip install -e ".[sklearn]"`; install Jupyter separately. `sd.evaluate` is new in this branch. `sd.see`, `.show()`, `.save()` and scientific reports are existing APIs.

In [ ]:
from pathlib import Path

from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

import seedl as sd

data = load_breast_cancer(as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.25, stratify=data.target, random_state=42
)
output = Path("artifacts/breast-cancer")
output.mkdir(parents=True, exist_ok=True)
print(dict(enumerate(data.target_names)))

## Train with sklearn, evaluate with SeeDL

Here, **0 = malignant** and **1 = benign**. We explicitly choose malignant (0) as the positive class for ROC and precision–recall. Never infer the meaning of a positive class from numeric ordering alone.

In [ ]:
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
model.fit(X_train, y_train)  # Explicit training in your code.
result = sd.evaluate(
    model,
    X_test,
    y_test,
    split="test",
    class_names=dict(enumerate(data.target_names)),
    positive_label=0,
    backend="svg",
)
result.save(output / "evaluation.html")
result

Accuracy alone can hide class imbalance. Precision measures correctness among predicted cases of a class; recall measures recovery among actual cases; F1 balances them. Support is the number of true examples per class. Macro metrics weight classes equally, weighted metrics use support. Undefined metrics remain unavailable instead of being replaced with zero.

In [ ]:
print(result.metrics)
print(result.available_plots)
roc = result.plot("roc")
roc.save(output / "roc.svg")
roc

The ROC AUC is computed from all binary scores before display sampling. Average precision is **not** a trapezoidal area under the displayed PR curve. Without `predict_proba`, a valid binary `decision_function` can supply uncalibrated margins; margins are not manufactured probabilities.

In [ ]:
result.plot("precision_recall").save(output / "precision-recall.html")
# Selecting a single view at evaluation time is also supported:
# sd.evaluate(model, X_test, y_test, split="test", plots="confusion_matrix")

## Inspect preprocessing and coefficients

Coefficients refer to standardized inputs, depend on scaling and encoding, and do not prove causality. For binary logistic regression they describe `classes_[1]` log-odds; choosing malignant as the positive **evaluation** label does not change the model coefficient orientation.

In [ ]:
inspection = sd.see(model, backend="svg")
inspection.save(output / "pipeline.html")
inspection

## Interpret on explicit held-out data

Permutation importance measures a score decrease when an original input column is shuffled through the entire pipeline. It can be unstable for correlated measurements and is not causal importance.

In [ ]:
importance = sd.feature_importance(
    model,
    X_test,
    y_test,
    method="permutation",
    scoring="accuracy",
    split="test",
    n_repeats=3,
    max_samples=128,
    backend="svg",
)
importance.save(output / "permutation.html")
print("Offline results:", output.resolve())

Multiclass classifiers produce per-class and macro/weighted reports. Binary ROC/PR plots are intentionally unavailable for multiclass models rather than relabeling a multiclass problem as binary. Source: [Wisconsin Diagnostic Breast Cancer](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset).